In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("silver layer") \
    .getOrCreate()

In [0]:
%sql
SHOW SCHEMAS IN shoplive;

In [0]:
customers_df = spark.table("shoplive.bronze.customers")

orders_df = spark.table("shoplive.bronze.orders")

products_df = spark.table("shoplive.bronze.products")

events_df = spark.table("shoplive.bronze.events")

In [0]:
display(products_df)

In [0]:
%sql
select count(*) from shoplive.bronze.customers

In [0]:
display(customers_df)

In [0]:
display(orders_df)

In [0]:
display(events_df)

In [0]:
from pyspark.sql.functions import split, col, regexp_replace

clean_df = customers_df.withColumn("first_name", split(col("name"), " ")[0]) \
       .withColumn("second_name", split(col("name"), " ")[1]) \
       .drop("name") \
       .withColumn("email", regexp_replace(col("email"), "(?<=.{2})[^@]+(?=@)", "***")) \
       .dropDuplicates(["customer_id"])
       

In [0]:
display(clean_df)

In [0]:
from pyspark.sql.functions import to_date, date_format

clean_events_df = events_df.dropDuplicates(["event_id"]) \
    .withColumn("event_date", to_date(col("event_ts"))) \
    .withColumn("event_time", date_format(col("event_ts"), "HH:mm:ss")) \
    .drop("event_ts")
    
display(clean_events_df)


In [0]:
from pyspark.sql.functions import to_date, date_format, col

clean_orders_df = orders_df.dropDuplicates(["order_id"]) \
    .withColumn("order_date", to_date(col("order_ts"))) \
    .withColumn("order_time", date_format(col("order_ts"), "HH:mm:ss")) \
    .drop("order_ts")

In [0]:
display(clean_orders_df)

In [0]:
clean_products_df = products_df.dropDuplicates(["product_id"])

In [0]:
display(clean_products_df)

In [0]:
clean_df.write.mode("overwrite").saveAsTable("shoplive.silver.customers")

clean_orders_df.write.mode("overwrite").saveAsTable("shoplive.silver.orders")

clean_products_df.write.mode("overwrite").saveAsTable("shoplive.silver.products")

clean_events_df.write.mode("overwrite").saveAsTable("shoplive.silver.events")